#Joins:
1.Inner join<br>
2.left<br>
3.right<br>
4.full<br>
5.left_semi<br>
6.left_anti<br>
7.cross



In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import to_date, col
customer_data = [
    (1, "John", "M", "New York"),
    (2, "Alice", "F", "Los Angeles"),
    (3, "Bob", "M", "Chicago"),
    (4, "Mary", "F", "Houston"),
    (5, "David", "M", "Phoenix")
]

customer_schema = StructType([
    StructField("cust_id", IntegerType(), True),
    StructField("cust_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True)
])

df_customer = spark.createDataFrame(customer_data, schema=customer_schema)
df_customer.show()
txn_data = [
    (101, 1, "2025-10-01", 250.50),
    (102, 2, "2025-10-03", 450.00),
    (103, 1, "2025-10-05", 180.75),
    (104, 4, "2025-09-29", 999.99),
    (105, 6, "2025-10-07", 120.00)   # Note: cust_id=6 not in customer table
]

txn_schema = StructType([
    StructField("txn_id", IntegerType(), True),
    StructField("cust_id", IntegerType(), True),
    StructField("txn_date", StringType(), True),
    StructField("amount", DoubleType(), True)
])

df_txn = spark.createDataFrame(txn_data, schema=txn_schema)
df_txn = df_txn.withColumn("txn_date", to_date(col("txn_date"), "yyyy-MM-dd"))
df_txn.show()

#Basic Join Syntax
- df1.join(df2, on="key_column", how="join_type")<br>
- on → column(s) to join on (string or list of column names)<br>
- how → type of join: "inner", "left", "right", "full", "left_semi", "left_anti", "cross"

In [0]:
#inner Join
df_customer.join(df_txn,df_customer.cust_id==df_txn.cust_id,how = "inner").display()

In [0]:
#Left outer join
df_customer.join(df_txn,on="cust_id",how = "left_outer").display()

In [0]:
#Right outer join
df_customer.join(df_txn,on="cust_id",how = "right_outer").display()

In [0]:
#Full outer join
df_customer.join(df_txn,on="cust_id",how = "full_outer").display()

In [0]:
#Left Semi join
#left_semi returns only the rows from the left DataFrame (df1) that have a matching key in the right DataFrame (df2).
df_customer.join(df_txn,on="cust_id",how = "left_semi").display()

In [0]:
#Left anti join
#left_anti returns only the rows from the left DataFrame (df1) that do not have a match in the right DataFrame (df2).
df_customer.join(df_txn,on="cust_id",how = "left_anti").display()

In [0]:
#Cross join
df_customer.crossJoin(df_txn).display()

In [0]:
#multi df join
from pyspark.sql.functions import *


customers = spark.createDataFrame([
    (1, "Rahul", "Mumbai"),
    (2, "Priya", "Delhi"),
    (3, "Arjun", "Chennai")
], ["cust_id", "name", "city"])

transactions = spark.createDataFrame([
    ("T01", 1, "P01", 1000),
    ("T02", 1, "P02", 800),
    ("T03", 2, "P01", 500)
], ["txn_id", "cust_id", "prod_id", "amount"])

products = spark.createDataFrame([
    ("P01", "Laptop", "Electronics"),
    ("P02", "Mobile", "Electronics"),
    ("P03", "Chair", "Furniture")
], ["prod_id", "prod_name", "category"])


final_df = customers.join(transactions, on="cust_id", how="inner") \
                    .join(products, on="prod_id", how="inner")

final_df.show()
